# LanceDB To Kayak Reranking

This notebook shows a concrete handoff between a vector database and Kayak.

Flow:

1. encode real text to ColBERT-style token vectors on CPU
2. store those multivectors in LanceDB
3. run a filtered LanceDB search to get a candidate set
4. convert the candidate rows into a Kayak `LateIndex`
5. run exact Mojo search and clause-text verification on that candidate set

Install for this notebook if needed:

```bash
uv add lancedb pyarrow pandas
```

or:

```bash
pixi add --pypi lancedb pyarrow pandas
```

In [1]:
import warnings

from pathlib import Path

import kayak
import lancedb
import numpy as np
import pyarrow as pa
from colbert.infra.config import ColBERTConfig
from colbert.modeling.checkpoint import Checkpoint
import torch

DEFAULT_MODEL_NAME = "colbert-ir/colbertv2.0"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA is not available.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="torch.cuda.amp.GradScaler is enabled, but CUDA is not available.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="resource_tracker: There appear to be .* leaked semaphore objects.*",
    category=UserWarning,
)

torch.set_num_threads(1)
torch.multiprocessing.set_sharing_strategy("file_system")

checkpoint = Checkpoint(
    DEFAULT_MODEL_NAME,
    colbert_config=ColBERTConfig(gpus=0),
    verbose=0,
)

def encode_query_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.queryFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()


def encode_document_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.docFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND

print("available_backends:", kayak.available_backends())
print("backend_info:", kayak.backend_info(BACKEND))
print("encoder_model:", DEFAULT_MODEL_NAME)

/tmp/kayak-lancedb-tools/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


available_backends: ('numpy_reference', 'mojo_exact_cpu')
backend_info: BackendInfo(name='mojo_exact_cpu', available=True, requires_mojo=True, query_layouts=('nested', 'flat_dim128'), index_layouts=('packed', 'hybrid_flat_dim128'), availability_reason='Kayak can invoke Mojo via: /Users/teilomillet/Code/kayak/.pixi/envs/default/bin/mojo')
encoder_model: colbert-ir/colbertv2.0


In [2]:
documents = [
    {
        "doc_id": "apple_m2_ultra",
        "category": "hardware",
        "text": "Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.",
    },
    {
        "doc_id": "nvidia_h100",
        "category": "hardware",
        "text": "NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.",
    },
    {
        "doc_id": "modular_mojo",
        "category": "software",
        "text": "Modular develops the Mojo programming language and compiler tools for AI systems.",
    },
    {
        "doc_id": "pixi_env_manager",
        "category": "tooling",
        "text": "Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.",
    },
    {
        "doc_id": "colbert_late_interaction",
        "category": "retrieval",
        "text": "ColBERT uses late interaction with token-level MaxSim scoring instead of a single document embedding.",
    },
]

documents

[{'doc_id': 'apple_m2_ultra',
  'category': 'hardware',
  'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'},
 {'doc_id': 'nvidia_h100',
  'category': 'hardware',
  'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'},
 {'doc_id': 'modular_mojo',
  'category': 'software',
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'},
 {'doc_id': 'pixi_env_manager',
  'category': 'tooling',
  'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'},
 {'doc_id': 'colbert_late_interaction',
  'category': 'retrieval',
  'text': 'ColBERT uses late interaction with token-level MaxSim scoring instead of a single document embedding.'}]

In [3]:
encoded_documents = [np.asarray(encode_document_text(doc["text"]), dtype=np.float32) for doc in documents]
print("document_vector_counts:", [int(v.shape[0]) for v in encoded_documents])
print("vector_dim:", int(encoded_documents[0].shape[1]))

[Apr 15, 12:30:10] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


document_vector_counts: [23, 24, 16, 28, 24]
vector_dim: 128


## Create A LanceDB Table With Multivectors

In [4]:
database_root = Path("/tmp/kayak-docs-lancedb-demo")
if database_root.exists():
    import shutil
    shutil.rmtree(database_root)

db = lancedb.connect(str(database_root))
vector_dim = int(encoded_documents[0].shape[1])

schema = pa.schema([
    pa.field("doc_id", pa.string()),
    pa.field("category", pa.string()),
    pa.field("text", pa.string()),
    pa.field("vector", pa.list_(pa.list_(pa.float32(), vector_dim))),
])

rows = []
for doc, vectors in zip(documents, encoded_documents, strict=True):
    rows.append({**doc, "vector": vectors.tolist()})

table = db.create_table("docs", data=rows, schema=schema, mode="overwrite")
table.count_rows()

[2026-04-15T10:30:10Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/kayak-docs-lancedb-demo/docs.lance, it will be created


5

## Stage 1 In LanceDB: Filter And Search

In [5]:
query_text = "which company develops the Mojo programming language?"
query_vectors = np.asarray(encode_query_text(query_text), dtype=np.float32)

coarse_rows = (
    table.search(query_vectors)
    .where("category IN ('software', 'tooling')")
    .limit(3)
    .to_arrow()
    .to_pylist()
)

[(row["doc_id"], round(float(row["_distance"]), 3), row["category"]) for row in coarse_rows]

[('modular_mojo', -19.043, 'software'), ('pixi_env_manager', -7.38, 'tooling')]

## Convert The Candidate Rows To A Kayak Index

In [6]:
query = kayak.query(query_vectors, text=query_text)

candidate_index = kayak.documents(
    [row["doc_id"] for row in coarse_rows],
    [np.asarray(row["vector"], dtype=np.float32) for row in coarse_rows],
    texts=[row["text"] for row in coarse_rows],
).pack()

print("candidate_layout:", candidate_index.layout)
print("candidate_document_count:", candidate_index.document_count)
print("candidate_total_vector_count:", candidate_index.total_vector_count)

candidate_layout: packed
candidate_document_count: 2
candidate_total_vector_count: 44


## Exact Kayak Search On The Candidate Set

In [7]:
documents_by_id = {doc["doc_id"]: doc for doc in documents}

def describe_hits(hits):
    return [
        {
            "doc_id": hit.doc_id,
            "score": round(hit.score, 3),
            "category": documents_by_id[hit.doc_id]["category"],
            "text": documents_by_id[hit.doc_id]["text"],
        }
        for hit in hits
    ]

exact_hits = kayak.search(query, candidate_index, k=2, backend=BACKEND)
describe_hits(exact_hits)

[{'doc_id': 'modular_mojo',
  'score': 25.855,
  'category': 'software',
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'},
 {'doc_id': 'pixi_env_manager',
  'score': 17.839,
  'category': 'tooling',
  'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'}]

## Stage-Aware Verification On The Same Candidate Set

In [8]:
plan = kayak.exact_full_scan_search_plan(
    final_k=2,
    candidate_k=candidate_index.document_count,
    stage3_verifier=kayak.clause_text_stage3_verifier_operator(),
)

result = kayak.search_with_plan(
    query,
    candidate_index,
    plan,
    backend=BACKEND,
)

print("final_hits:")
print(describe_hits(result.hits))

print("candidate_stage_profile:", result.candidate_stage.profile)
print("stage2_profile:", result.stage2)
print("stage3_profile:", result.stage3_verifier)

final_hits:
[{'doc_id': 'modular_mojo', 'score': 28.955, 'category': 'software', 'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'}, {'doc_id': 'pixi_env_manager', 'score': 18.339, 'category': 'tooling', 'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'}]
candidate_stage_profile: SearchStageProfile(stage_name='exact_full_scan', input_hit_count=2, output_hit_count=2, query_vector_count=32, document_count=2, document_vector_count=44, document_text_count=0, materialized_artifacts=())
stage2_profile: SearchStageProfile(stage_name='noop_topk', input_hit_count=2, output_hit_count=2, query_vector_count=0, document_count=2, document_vector_count=0, document_text_count=0, materialized_artifacts=())
stage3_profile: SearchStageProfile(stage_name='clause_text', input_hit_count=2, output_hit_count=2, query_vector_count=0, document_count=2, document_vector_count=0, document_text_count=2, materialize

## Exact Baseline On The Full LanceDB Slice

In [9]:
full_rows = table.to_arrow().to_pylist()

full_index = kayak.documents(
    [row["doc_id"] for row in full_rows],
    [np.asarray(row["vector"], dtype=np.float32) for row in full_rows],
    texts=[row["text"] for row in full_rows],
).pack()

full_exact_hits = kayak.search(query, full_index, k=3, backend=BACKEND)
describe_hits(full_exact_hits)

[{'doc_id': 'modular_mojo',
  'score': 25.855,
  'category': 'software',
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'},
 {'doc_id': 'pixi_env_manager',
  'score': 17.839,
  'category': 'tooling',
  'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'},
 {'doc_id': 'nvidia_h100',
  'score': 9.308,
  'category': 'hardware',
  'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'}]

In [10]:
candidate_ids = {row["doc_id"] for row in coarse_rows}
exact_ids = {hit.doc_id for hit in full_exact_hits}
candidate_overlap = len(candidate_ids & exact_ids) / len(exact_ids)
candidate_ids, exact_ids, candidate_overlap

({'modular_mojo', 'pixi_env_manager'},
 {'modular_mojo', 'nvidia_h100', 'pixi_env_manager'},
 0.6666666666666666)